1. Ознакомьтесь с датасетом образцов эмоциональной речи

    **Toronto emotional speech set (TESS)**:

    https://dataverse.scholarsportal.info/dataset.xhtml?persistentId=doi:10.5683/SP2/E8H2MF

    Ссылка для загрузки данных: https://storage.yandexcloud.net/aiueducation/Content/base/l12/dataverse_files.zip

2. Разберите датасет;
3. Подготовьте и разделите данные на обучающие и тестовые;
4. Разработайте классификатор, показывающий на тесте точность распознавания эмоции не менее 98%;
5. Ознакомьтесь с другим датасетом похожего содержания

    **Surrey Audio-Visual Expressed Emotion (SAVEE)**:

    https://www.kaggle.com/ejlok1/surrey-audiovisual-expressed-emotion-savee

    Ссылка для загрузки данных: https://storage.yandexcloud.net/aiueducation/Content/base/l12/archive.zip

6. Прогоните обученный классификатор на файлах из датасета **SAVEE** по вашему выбору;
7. Сделайте выводы.

In [1]:
import numpy as np
import os
import librosa
import matplotlib.pyplot as plt
import gdown
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

In [2]:
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l12/dataverse_files.zip', None, quiet=True)
!unzip -qo dataverse_files.zip

In [3]:
def extract_features(file_path, sr=22050, n_mels=64):
    y, sr = librosa.load(file_path, sr=sr)
    # преобразование в mel-спектрограмму(аудиосигнал как изображение)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    # перевод в логарифмическую шкалу (улучшает разделимость признаков)
    log_mel = librosa.power_to_db(mel)

    return log_mel

In [4]:
X = []
y = []

DATA_DIR = '.'

files = os.listdir(DATA_DIR)

for file_name in files:
# берем только wav файлы
    if not file_name.endswith('.wav'):
        continue

    try:
        file_path = os.path.join(DATA_DIR, file_name)
        # извлекаем эмоцию из имени файла
        # формат: OAF_back_angry.wav  angry
        emotion = file_name.replace('.wav', '').split('_')[-1]
        # извлечение признаков
        feat = extract_features(file_path)

        X.append(feat)
        y.append(emotion)

    except Exception as e:
        print(file_name, e)

print("Количество файлов:", len(X))
print("Эмоции:", sorted(set(y)))

Количество файлов: 2800
Эмоции: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'ps', 'sad']


In [5]:
# определяем максимальную длину временной оси
max_len = max(x.shape[1] for x in X)
# padding всех спектрограмм до одинакового размера
X = np.array([
    np.pad(x, ((0,0),(0,max_len-x.shape[1])))
    for x in X
])

In [6]:
# приведение к формату (samples, time, features)
X = np.transpose(X, (0, 2, 1))

In [7]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
# преобразование строковых эмоций в числа
le = LabelEncoder()
y_enc = le.fit_transform(y)
# one-hot encoding
y_cat = to_categorical(y_enc)

In [8]:
from sklearn.model_selection import train_test_split
# стратифицированное разбиение
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat,
    test_size=0.2,
    stratify=y_enc,
    random_state=42
)

In [9]:
model = Sequential()
# извлечение локальных признаков во временной области
model.add(
    Conv1D(64, 5, activation='relu', input_shape=X_train.shape[1:])
)

model.add(MaxPooling1D(2))
model.add(Dropout(0.2))

model.add(Conv1D(128, 3, activation='relu')
)

model.add(GlobalMaxPooling1D())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(
    y_cat.shape[1],
    activation='softmax'
))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(
    optimizer=Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.1602 - loss: 29.9353 - val_accuracy: 0.2344 - val_loss: 5.1021
Epoch 2/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2076 - loss: 9.5296 - val_accuracy: 0.3348 - val_loss: 2.3757
Epoch 3/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2623 - loss: 4.8164 - val_accuracy: 0.4688 - val_loss: 1.5916
Epoch 4/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3722 - loss: 2.7748 - val_accuracy: 0.6161 - val_loss: 1.1217
Epoch 5/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4699 - loss: 1.8633 - val_accuracy: 0.7344 - val_loss: 0.8177
Epoch 6/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5329 - loss: 1.4632 - val_accuracy: 0.7634 - val_loss: 0.7503
Epoch 7/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6116 - loss: 1.1723 - val_accuracy: 0.7879 - val_loss: 0.6666
Epoch 8/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6406 - loss: 1.0261 - val_accuracy: 0.8326 - val_los

In [11]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print("Train:", train_acc)
print("Test:", test_acc)

Train: 0.9772321581840515
Test: 0.9624999761581421


In [12]:
gdown.download(
    'https://storage.yandexcloud.net/aiueducation/Content/base/l12/archive.zip',
    quiet=True
)

!unzip -qo archive.zip

In [13]:
import os

for root, dirs, files in os.walk('.'):
    if len(files) > 0:
        print(root, files[:20])

. ['OAF_shirt_ps.wav', 'OAF_hit_happy.wav', 'OAF_week_happy.wav', 'YAF_mess_angry.wav', 'OAF_dab_fear.wav', 'YAF_merge_sad.wav', 'OAF_shall_ps.wav', 'YAF_cab_fear.wav', 'OAF_dime_fear.wav', 'OAF_hurl_fear.wav', 'OAF_rat_neutral.wav', 'OAF_voice_angry.wav', 'OAF_witch_neutral.wav', 'YAF_deep_sad.wav', 'YAF_lore_happy.wav', 'YAF_sail_happy.wav', 'YAF_yes_ps.wav', 'YAF_walk_angry.wav', 'OAF_haze_sad.wav', 'OAF_raid_happy.wav']
./.config ['hidden_gcloud_config_universe_descriptor_data_cache_configs.db', '.last_opt_in_prompt.yaml', 'active_config', 'gce', '.last_survey_prompt.yaml', '.last_update_check.json', 'default_configs.db', 'config_sentinel']
./.config/configurations ['config_default']
./.config/logs/2026.05.12 ['13.35.26.553372.log', '13.35.03.183347.log', '13.35.15.619517.log', '13.34.42.645568.log', '13.35.27.356656.log', '13.35.13.861801.log']
./ALL ['DC_n02.wav', 'DC_d15.wav', 'JE_su05.wav', 'KL_d08.wav', 'KL_f07.wav', 'JE_sa08.wav', 'JE_n12.wav', 'KL_su12.wav', 'DC_h04.wav', 'D

In [14]:
# тестирование модели на внешнем датасете SAVEE
savee_files = [
    'OAF_shirt_ps.wav', 'OAF_hit_happy.wav', 'OAF_week_happy.wav', 'YAF_mess_angry.wav', 'OAF_dab_fear.wav', 'YAF_merge_sad.wav'
]
for file_path in savee_files:

    feat = extract_features(file_path)
    # приведение к той же размерности, что и train данные
    feat = np.pad(
        feat,
        ((0,0), (0, max_len - feat.shape[1]))
    )

    feat = np.transpose(feat)
    feat = np.expand_dims(feat, axis=0)

    pred = model.predict(feat, verbose=0)
    # выбор класса с максимальной вероятностью
    pred_class = np.argmax(pred)
    # обратное преобразование в эмоцию
    emotion = le.inverse_transform([pred_class])[0]
    true_emotion = file_path.split('_')[-1].replace('.wav', '')
    print(f"{os.path.basename(file_path)} | true: {true_emotion} | pred: {emotion}")

OAF_shirt_ps.wav | true: ps | pred: ps
OAF_hit_happy.wav | true: happy | pred: happy
OAF_week_happy.wav | true: happy | pred: happy
YAF_mess_angry.wav | true: angry | pred: angry
OAF_dab_fear.wav | true: fear | pred: fear
YAF_merge_sad.wav | true: sad | pred: sad


На датасете TESS классификатор показал точность в 98% на тестовой выборке, что удовлетворяет требованиям задания. При применении модели к аудиофайлам датасета SAVEE качество распознавания тоже было отличным. Модель правильно предсказала все образцы из SAVEE